In [1]:
import warnings
warnings.filterwarnings("ignore")

from filings_rag.sec_chunking import parse_sec_html, build_chunks, chunk_type_counts
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")

In [3]:
path = "../data/raw/aapl_FY2025_10K.html"
elements = parse_sec_html(path)

for e in elements:
    if e.element_type == "heading":
        print(f"[{e.metadata.get('heading_level'):5}] part={e.part!r:10} item={e.item!r:35} | {e.text[:90]}")

[section] part=None       item=None                                | UNITED STATES
[section] part=None       item=None                                | SECURITIES AND EXCHANGE COMMISSION
[section] part=None       item=None                                | Washington, D.C. 20549
[section] part=None       item=None                                | FORM 10-K
[section] part=None       item=None                                | ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
[section] part=None       item=None                                | For the fiscal year ended September 27 , 2025
[section] part=None       item=None                                | or
[section] part=None       item=None                                | ☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
[section] part=None       item=None                                | For the transition period from to .
[section] part=None       item=Non

In [4]:
#parse and chunk 10 fillings into one combined corpus
filings = {
    "aapl_FY2025_10K":  ("APPLE",     "FY2025"),
    "amzn_FY2025_10K":  ("AMAZON",    "FY2025"),
    "dell_FY2026_10K":  ("DELL",      "FY2026"),
    "goog_FY2025_10K":  ("ALPHABET",  "FY2025"),
    "meta_FY2025_10K":  ("META",      "FY2025"),
    "msft_FY2026_10K":  ("MICROSOFT", "FY2026"),
    "nvda_FY2024_10K":  ("NVIDIA",    "FY2024"),
    "nvda_FY2025_10K":  ("NVIDIA",    "FY2025"),
    "nvda_FY2026_10K":  ("NVIDIA",    "FY2026"),
    "pltr_FY2025_10K":  ("PALANTIR",  "FY2025"),
}

all_chunks = []

for key, (company, period) in filings.items():
    path = f"../data/raw/{key}.html"
    elements = parse_sec_html(path)
    chunks = build_chunks(
        elements,
        tok,
        filing_key=key,
        doc_label=f"{company} 10-K {period}",
        company=company,
        filing_type="10-K",
        fiscal_period=period,
    )
    all_chunks.extend(chunks)
    print(f"{key}: {len(chunks)} chunks")

print(f"\ntotal: {len(all_chunks)} chunks")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (820 > 512). Running this sequence through the model will result in indexing errors


aapl_FY2025_10K: 245 chunks
amzn_FY2025_10K: 316 chunks
dell_FY2026_10K: 515 chunks
goog_FY2025_10K: 454 chunks
meta_FY2025_10K: 487 chunks
msft_FY2026_10K: 378 chunks
nvda_FY2024_10K: 375 chunks
nvda_FY2025_10K: 384 chunks
nvda_FY2026_10K: 356 chunks
pltr_FY2025_10K: 486 chunks

total: 3996 chunks


In [5]:
print(all_chunks[0]["chunk_id"])
print(all_chunks[-1]["chunk_id"])

aapl_FY2025_10K:4:1
pltr_FY2025_10K:1537:1


In [ ]:
#write the corpus this notebook just built. schema and key order match
#all_chunks_v2.jsonl exactly - index, chunk_id, doc, section, text - so the next cell
#can compare them byte for byte. chunk_id was missing from the earlier version of this
#cell, which is why its output could never be diffed against the corpus in use.
import json

with open("../data/gold/all_chunks_check.jsonl", "w") as f:
    for i, c in enumerate(all_chunks):
        f.write(json.dumps({
            "index": i,
            "chunk_id": c["chunk_id"],
            "doc": c["doc"],
            "section": c.get("section"),
            "text": c["text"],
        }) + "\n")

print(f"wrote {len(all_chunks)} chunks -> ../data/gold/all_chunks_check.jsonl")

In [ ]:
#verify: does this run reproduce the corpus the evaluation actually uses?
#counting chunks is not enough - a table serialised differently would still give the
#same count - so compare line by line and hash the whole file.
import hashlib

check_path = "../data/gold/all_chunks_check.jsonl"
ref_path   = "../data/gold/all_chunks_v2.jsonl"

check = open(check_path, encoding="utf-8").read().splitlines()
ref   = open(ref_path,   encoding="utf-8").read().splitlines()

h = lambda lines: hashlib.sha256("\n".join(lines).encode("utf-8")).hexdigest()
identical = h(check) == h(ref)

print(f"{'chunks written':26} {len(check)}")
print(f"{'chunks in all_chunks_v2':26} {len(ref)}")
print(f"{'byte-identical':26} {identical}")

if not identical:
    if len(check) != len(ref):
        print(f"\ncount differs by {len(check) - len(ref)} - the chunker itself changed")
    diffs = [i for i, (a, b) in enumerate(zip(check, ref)) if a != b]
    print(f"{'differing lines':26} {len(diffs)} of {min(len(check), len(ref))}")
    for i in diffs[:3]:
        a, b = json.loads(check[i]), json.loads(ref[i])
        print(f"\n  line {i}  chunk_id {a['chunk_id']} vs {b['chunk_id']}")
        for k in ("doc", "section", "text"):
            if a.get(k) != b.get(k):
                print(f"    {k}:")
                print(f"      this run: {str(a.get(k))[:120]}")
                print(f"      v2      : {str(b.get(k))[:120]}")
else:
    print("\nthe chunker reproduces all_chunks_v2 exactly - the corpus under evaluation")
    print("is regenerable from data/raw, not a one-off artefact.")